# Business Analytics Project — Flipkart Sales & Business Performance Analytics

**Dataset:** `flipkart_sales_enriched.csv`  
**Project report:** `Flipkart_Business_Analytics_Project_Report.docx`

This notebook follows the structure and methods in the supplied Flipkart Business Analytics report. It recalculates the transaction-based analyses from the supplied CSV and clearly labels simulated/proposed sections where the report says the historical dataset does not contain the required experimental fields.

**Dataset:** 1,000 transaction records. **Random seed:** 42.


In [ ]:
# 1. Imports
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded.")


In [ ]:
# 2. Load dataset
csv_candidates = [
    "/mnt/data/flipkart_sales_enriched.csv",
    "flipkart_sales_enriched.csv",
    "flipkart_sales_enriched(1).csv"
]

csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except Exception:
        raise FileNotFoundError("Dataset CSV not found. Upload flipkart_sales_enriched.csv.")

df = pd.read_csv(csv_path)
df["Order Date"] = pd.to_datetime(df["Order Date"])

print("Loaded:", csv_path)
print("Shape:", df.shape)
display(df.head())


## 3. Dataset Description & Preprocessing

In [ ]:
# Dataset overview
display(pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": [df[c].nunique() for c in df.columns]
}))

print("Duplicate rows:", df.duplicated().sum())
print("Date range:", df["Order Date"].min().date(), "to", df["Order Date"].max().date())
print("Negative quantity:", int((df["Quantity Sold"] < 0).sum()))
print("Negative sales:", int((df["Total Sales (INR)"] < 0).sum()))
print("Negative profit:", int((df["Profit (INR)"] < 0).sum()))

# Useful derived fields
df["Month_Num"] = df["Order Date"].dt.month
df["Year_Calc"] = df["Order Date"].dt.year
df["Month_Calc"] = df["Order Date"].dt.to_period("M").astype(str)


## 4. Exploratory Data Analysis

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Categories:", df["Category"].nunique())
print("Payment methods:", df["Payment Method"].nunique())
print("Customer segments:", df["Customer Segment"].nunique())
print("Regions:", df["Region"].nunique())
print("Products:", df["Product Name"].nunique())

for col in ["Category", "Payment Method", "Customer Segment", "Region"]:
    print(f"\n--- {col} ---")
    display(df[col].value_counts().rename("Orders").to_frame())


In [ ]:
# Sales by Category
category_summary = df.groupby("Category").agg(
    Orders=("Order ID", "count"),
    Sales=("Total Sales (INR)", "sum"),
    Profit=("Profit (INR)", "sum"),
    Avg_Rating=("Customer Rating", "mean")
).sort_values("Sales", ascending=False)
display(category_summary)

fig, ax = plt.subplots(figsize=(9,5))
category_summary["Sales"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Sales by Category")
ax.set_xlabel("Total Sales (INR)")
plt.tight_layout()
plt.show()

# Profit by Region
region_summary = df.groupby("Region").agg(
    Orders=("Order ID", "count"),
    Sales=("Total Sales (INR)", "sum"),
    Profit=("Profit (INR)", "sum")
).sort_values("Profit", ascending=False)
display(region_summary)

fig, ax = plt.subplots(figsize=(9,5))
region_summary["Profit"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Profit by Region")
ax.set_xlabel("Profit (INR)")
plt.tight_layout()
plt.show()


In [ ]:
# Sales and profit distributions
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(df["Total Sales (INR)"], bins=30)
ax.set_title("Sales Histogram")
ax.set_xlabel("Total Sales (INR)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(df["Profit (INR)"], bins=30)
ax.set_title("Profit Histogram")
ax.set_xlabel("Profit (INR)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,4))
ax.boxplot(df["Total Sales (INR)"], vert=False)
ax.set_title("Sales Box Plot")
ax.set_xlabel("Total Sales (INR)")
plt.tight_layout()
plt.show()


In [ ]:
# Payment Method distribution
payment_counts = df["Payment Method"].value_counts()

fig, ax = plt.subplots(figsize=(7,7))
ax.pie(payment_counts.values, labels=payment_counts.index, autopct="%1.1f%%")
ax.set_title("Payment Method Distribution")
plt.tight_layout()
plt.show()

# Price vs Sales
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(df["Price (INR)"], df["Total Sales (INR)"], alpha=0.5)
ax.set_title("Price vs Total Sales")
ax.set_xlabel("Price (INR)")
ax.set_ylabel("Total Sales (INR)")
plt.tight_layout()
plt.show()


## 5. Descriptive Statistical Analysis

In [ ]:
variables = [
    "Price (INR)", "Quantity Sold", "Total Sales (INR)",
    "Customer Rating", "Profit (INR)", "Discount %"
]

rows = []
for col in variables:
    s = df[col].dropna()
    rows.append({
        "Variable": col,
        "Mean": s.mean(),
        "Median": s.median(),
        "Mode": s.mode().iloc[0],
        "Min": s.min(),
        "Max": s.max(),
        "Range": s.max() - s.min(),
        "Variance": s.var(),
        "Std Dev": s.std()
    })

desc = pd.DataFrame(rows).set_index("Variable")
display(desc)

print("Total Sales: ₹{:,.2f}".format(df["Total Sales (INR)"].sum()))
print("Total Profit: ₹{:,.2f}".format(df["Profit (INR)"].sum()))
print("Average Order Sales: ₹{:,.2f}".format(df["Total Sales (INR)"].mean()))
print("Average Profit: ₹{:,.2f}".format(df["Profit (INR)"].mean()))
print("Average Quantity Sold: {:.2f}".format(df["Quantity Sold"].mean()))


## 6. Distribution Analysis

In [ ]:
sales = df["Total Sales (INR)"]

print("Mean:", sales.mean())
print("Median:", sales.median())
print("Std Dev:", sales.std())
print("Skewness:", sales.skew())

if sales.skew() > 0:
    print("Interpretation: positive/right skew — some high-value orders pull the mean upward.")
elif sales.skew() < 0:
    print("Interpretation: negative/left skew.")
else:
    print("Interpretation: approximately symmetric.")


## 7. Correlation Analysis

In [ ]:
corr_cols = [
    "Price (INR)", "Quantity Sold", "Total Sales (INR)",
    "Customer Rating", "Profit (INR)", "Discount %"
]
corr = df[corr_cols].corr()
display(corr)

pairs = [
    ("Price (INR)", "Total Sales (INR)"),
    ("Quantity Sold", "Total Sales (INR)"),
    ("Discount %", "Total Sales (INR)"),
    ("Customer Rating", "Total Sales (INR)"),
    ("Price (INR)", "Profit (INR)")
]

for a, b in pairs:
    print(f"{a} vs {b}: {df[a].corr(df[b]):.3f}")

fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(corr, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Matrix")
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 8. Regression Analysis

In [ ]:
# Model: Total Sales = intercept + coefficient × Price
X = df[["Price (INR)"]]
y = df["Total Sales (INR)"]

reg = LinearRegression().fit(X, y)
pred = reg.predict(X)

r2 = reg.score(X, y)
rmse = np.sqrt(np.mean((y - pred) ** 2))

print(f"Intercept = {reg.intercept_:,.2f}")
print(f"Coefficient = {reg.coef_[0]:,.4f}")
print(f"R² = {r2:.4f}")
print(f"RMSE = ₹{rmse:,.2f}")
print(f"Equation: Sales = {reg.intercept_:,.2f} + {reg.coef_[0]:.4f} × Price")

fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(X["Price (INR)"], y, alpha=0.45)
order = np.argsort(X["Price (INR)"].values)
ax.plot(X["Price (INR)"].values[order], pred[order])
ax.set_title("Linear Regression: Price vs Total Sales")
ax.set_xlabel("Price (INR)")
ax.set_ylabel("Total Sales (INR)")
plt.tight_layout()
plt.show()


## 9. Business Intelligence Dashboard — KPI Data

In [ ]:
kpis = pd.DataFrame({
    "KPI": [
        "Total Sales", "Total Profit", "Orders",
        "Average Order Value", "Average Rating", "Average Discount"
    ],
    "Value": [
        df["Total Sales (INR)"].sum(),
        df["Profit (INR)"].sum(),
        df["Order ID"].nunique(),
        df["Total Sales (INR)"].mean(),
        df["Customer Rating"].mean(),
        df["Discount %"].mean()
    ]
})
display(kpis)

monthly_sales = df.set_index("Order Date")["Total Sales (INR)"].resample("MS").sum()
fig, ax = plt.subplots(figsize=(11,5))
monthly_sales.plot(ax=ax)
ax.set_title("Monthly Sales")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Probability and Probability Distributions

In [ ]:
events = pd.DataFrame({
    "Event": [
        "Sales > ₹1,00,000",
        "Quantity Sold >= 4",
        "Customer Rating >= 4"
    ],
    "Estimated Probability": [
        (df["Total Sales (INR)"] > 100000).mean(),
        (df["Quantity Sold"] >= 4).mean(),
        (df["Customer Rating"] >= 4).mean()
    ]
})
display(events)

# Distribution demonstrations
rng = np.random.default_rng(42)
normal_sample = rng.normal(sales.mean(), sales.std(), 1000)
uniform_sample = rng.uniform(sales.min(), sales.max(), 1000)
exponential_sample = rng.exponential(sales.mean(), 1000)

for sample, title in [
    (normal_sample, "Normal Distribution"),
    (uniform_sample, "Uniform Distribution"),
    (exponential_sample, "Exponential Distribution")
]:
    fig, ax = plt.subplots(figsize=(9,5))
    ax.hist(sample, bins=30)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


## 11. Decision Making Under Uncertainty

In [ ]:
decision = pd.DataFrame({
    "Alternative": ["Conservative launch", "Aggressive launch"],
    "P(High)": [0.7, 0.4],
    "High Payoff": [450000, 100000],
    "P(Low)": [0.3, 0.6],
    "Low Payoff": [120000, 550000]
})
decision["Expected Value"] = (
    decision["P(High)"] * decision["High Payoff"] +
    decision["P(Low)"] * decision["Low Payoff"]
)
display(decision)
print("Highest expected value:", decision.loc[decision["Expected Value"].idxmax(), "Alternative"])


## 12. Sampling and Sampling Distribution

In [ ]:
rng = np.random.default_rng(42)
sampling_results = []

for n_sample in [30, 100]:
    sample_means = np.array([
        rng.choice(sales.values, size=n_sample, replace=False).mean()
        for _ in range(1000)
    ])
    theoretical_se = sales.std(ddof=1) / np.sqrt(n_sample)
    sampling_results.append({
        "Sample Size": n_sample,
        "Mean of Sample Means": sample_means.mean(),
        "SD of Sample Means": sample_means.std(ddof=1),
        "Theoretical SE": theoretical_se
    })

sampling_table = pd.DataFrame(sampling_results)
display(sampling_table)

fig, ax = plt.subplots(figsize=(9,5))
ax.hist([
    rng.choice(sales.values, size=30, replace=False).mean()
    for _ in range(1000)
], bins=30)
ax.set_title("Sampling Distribution of Mean Sales (n=30)")
ax.set_xlabel("Sample Mean Sales (INR)")
plt.tight_layout()
plt.show()


## 13. Confidence Interval Estimation

In [ ]:
mean_sales = sales.mean()
se = stats.sem(sales)
n = len(sales)

ci_rows = []
for level in [0.90, 0.95, 0.99]:
    lower, upper = stats.t.interval(level, df=n-1, loc=mean_sales, scale=se)
    ci_rows.append([f"{level:.0%}", lower, upper])

ci = pd.DataFrame(ci_rows, columns=["Confidence", "Lower", "Upper"])
display(ci)


## 14. Bootstrapping

In [ ]:
rng = np.random.default_rng(42)
bootstrap_medians = np.array([
    np.median(rng.choice(sales.values, size=len(sales), replace=True))
    for _ in range(1000)
])

boot_lower, boot_upper = np.percentile(bootstrap_medians, [2.5, 97.5])

print(f"Observed median: ₹{sales.median():,.2f}")
print("Bootstrap repetitions: 1000")
print(f"Bootstrap 95% lower: ₹{boot_lower:,.2f}")
print(f"Bootstrap 95% upper: ₹{boot_upper:,.2f}")

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(bootstrap_medians, bins=30)
ax.axvline(boot_lower, linestyle="--")
ax.axvline(boot_upper, linestyle="--")
ax.set_title("Bootstrap Distribution of Median Sales")
plt.tight_layout()
plt.show()


## 15. Hypothesis Testing

In [ ]:
# H0: mean sales = ₹75,000
t_stat, p_value = stats.ttest_1samp(sales, popmean=75000)

print(f"t-statistic = {t_stat:.3f}")
print(f"p-value = {p_value:.4f}")
print("Decision:", "Reject H0" if p_value < 0.05 else "Fail to reject H0")


## 16. A/B Testing — Simulated

In [ ]:
# The supplied transaction data do not contain website impressions/conversions.
# Therefore, this section uses simulated group-level conversion rates as stated in the report.

n_A = n_B = 1000
conv_A = 113
conv_B = 123

rate_A = conv_A / n_A
rate_B = conv_B / n_B

p_pool = (conv_A + conv_B) / (n_A + n_B)
se = np.sqrt(p_pool * (1-p_pool) * (1/n_A + 1/n_B))
z = (rate_B - rate_A) / se
p = 2 * (1 - stats.norm.cdf(abs(z)))

ab = pd.DataFrame({
    "Variant": ["A", "B"],
    "Visitors": [n_A, n_B],
    "Conversions": [conv_A, conv_B],
    "Conversion Rate": [rate_A, rate_B]
})
display(ab)

print(f"z statistic = {z:.2f}")
print(f"p-value = {p:.4f}")
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")


## 17. Time Series Analysis

In [ ]:
monthly = df.set_index("Order Date")["Total Sales (INR)"].resample("MS").sum()

print("Monthly periods:", len(monthly))
display(monthly.to_frame("Sales"))

# Final three months held out as test set
train = monthly.iloc[:-3]
test = monthly.iloc[-3:]

# Additive/damped exponential smoothing
ets = ExponentialSmoothing(
    train,
    trend="add",
    damped_trend=True
).fit()

ets_forecast = ets.forecast(3)

# ARIMA(1,1,1)
arima = ARIMA(train, order=(1,1,1)).fit()
arima_forecast = arima.forecast(3)

forecast = pd.DataFrame({
    "Actual": test.values,
    "Exponential Smoothing": ets_forecast.values,
    "ARIMA(1,1,1)": arima_forecast.values
}, index=test.index)

display(forecast)

fig, ax = plt.subplots(figsize=(11,5))
train.plot(ax=ax, label="Train")
test.plot(ax=ax, label="Actual Test")
ets_forecast.plot(ax=ax, label="Exponential Smoothing")
arima_forecast.plot(ax=ax, label="ARIMA")
ax.set_title("Monthly Sales Forecast")
ax.set_ylabel("Sales (INR)")
ax.legend()
plt.tight_layout()
plt.show()


## 18. Forecast Accuracy

In [ ]:
def metrics(actual, predicted):
    mae = np.mean(np.abs(actual - predicted))
    mse = np.mean((actual - predicted) ** 2)
    rmse = np.sqrt(mse)
    return mae, mse, rmse

ets_mae, ets_mse, ets_rmse = metrics(test.values, ets_forecast.values)
arima_mae, arima_mse, arima_rmse = metrics(test.values, arima_forecast.values)

accuracy = pd.DataFrame({
    "Model": ["Exponential Smoothing", "ARIMA(1,1,1)"],
    "MAE": [ets_mae, arima_mae],
    "MSE": [ets_mse, arima_mse],
    "RMSE": [ets_rmse, arima_rmse]
})
display(accuracy)
print("Preferred model by lower RMSE:", accuracy.loc[accuracy["RMSE"].idxmin(), "Model"])


## 19. Optimization Modeling

In [ ]:
# Illustrative category-capacity optimization.
# Resource coefficients are planning assumptions; profit coefficients come from observed category averages.

cat = df.groupby("Category").agg(
    avg_profit=("Profit (INR)", "mean"),
    avg_qty=("Quantity Sold", "mean")
).reset_index()

display(cat)

# Use scipy.optimize.linprog if available.
from scipy.optimize import linprog

# Illustrative resource coefficients, one row per category.
# These coefficients represent labor and machine units per planned bundle.
resource_labor = np.array([2.8, 2.7, 2.9, 3.0, 2.6])
resource_machine = np.array([1.8, 1.7, 1.9, 1.6, 1.5])

# Match category order to the displayed category list.
cat_order = ["Beauty", "Books", "Clothing", "Electronics", "Home & Kitchen"]
cat2 = cat.set_index("Category").loc[cat_order]

profits = cat2["avg_profit"].values

result = linprog(
    c=-profits,
    A_ub=np.vstack([resource_labor, resource_machine]),
    b_ub=[80, 55],
    bounds=[(0, None)] * len(cat_order),
    method="highs"
)

if result.success:
    opt = pd.DataFrame({
        "Category": cat_order,
        "Optimal Units": result.x,
        "Avg Profit/Unit": profits,
        "Profit Contribution": result.x * profits
    })
    display(opt)
    print(f"Objective value = ₹{-result.fun:,.2f}")
else:
    print("Optimization failed:", result.message)


## 20. Simulation Modeling

In [ ]:
# Demand/revenue scenario simulation: 1,000 trials per scenario.
rng = np.random.default_rng(42)

scenario_specs = {
    "Conservative": (0.80, 0.15),
    "Base": (1.00, 0.20),
    "Aggressive": (1.20, 0.25)
}

base_monthly_sales = monthly.mean()
sim_rows = []

for scenario, (multiplier, cv) in scenario_specs.items():
    sigma = np.sqrt(np.log(1 + cv**2))
    mu = np.log(base_monthly_sales * multiplier) - 0.5 * sigma**2
    revenues = rng.lognormal(mean=mu, sigma=sigma, size=1000)

    sim_rows.append({
        "Scenario": scenario,
        "Mean Revenue Index": revenues.mean(),
        "P5": np.percentile(revenues, 5),
        "P95": np.percentile(revenues, 95)
    })

simulation = pd.DataFrame(sim_rows)
display(simulation)


## 21. ANOVA

In [ ]:
# One-way ANOVA across product categories
groups = [
    group["Total Sales (INR)"].values
    for _, group in df.groupby("Category")
]

f_stat, p_value = stats.f_oneway(*groups)

print(f"F-statistic = {f_stat:.2f}")
print(f"p-value = {p_value:.4f}")
print("Decision:", "Reject H0" if p_value < 0.05 else "Fail to reject H0")
print("H0: all category mean sales are equal.")
print("H1: at least one category mean differs.")


## 22. Experimental Design

### Proposed controlled experiment

- **Factor 1:** Discount level — Low / High
- **Factor 2:** Customer segment — Online / Retail / Wholesale
- **Response:** Total Sales (INR)
- **Experimental unit:** Individual transaction/order
- **Procedure:** Randomly assign customers/orders to factor combinations in a controlled campaign and measure sales over the same time window.

The supplied transaction data are observational, so this section describes a future controlled experiment rather than claiming a historical experiment.


## 23. Factorial Experiment

In [ ]:
# Exploratory 2-factor analysis using the historical data.
# Discount level is split at the observed median, following the report.
median_discount = df["Discount %"].median()
factorial_df = df.copy()
factorial_df["Discount Level"] = np.where(
    factorial_df["Discount %"] >= median_discount, "High", "Low"
)

cell_summary = factorial_df.groupby(
    ["Discount Level", "Customer Segment"]
)["Total Sales (INR)"].mean().reset_index()

display(cell_summary)

factorial_model = smf.ols(
    'Q("Total Sales (INR)") ~ C(Q("Discount Level")) + C(Q("Customer Segment")) + C(Q("Discount Level")):C(Q("Customer Segment"))',
    data=factorial_df
).fit()

anova_table = sm.stats.anova_lm(factorial_model, typ=2)
display(anova_table)

print("These are exploratory associations from observational data, not causal experimental effects.")


## 24. Business Insights and Findings

In [ ]:
top_category = df.groupby("Category")["Total Sales (INR)"].sum().idxmax()
top_category_sales = df.groupby("Category")["Total Sales (INR)"].sum().max()

segment_avg = df.groupby("Customer Segment")["Total Sales (INR)"].mean().sort_values(ascending=False)
top_segment = segment_avg.index[0]

year_avg = df.groupby("Year_Calc")["Total Sales (INR)"].mean()

print(f"Total dataset revenue: ₹{df['Total Sales (INR)'].sum():,.2f}")
print(f"Total dataset profit: ₹{df['Profit (INR)'].sum():,.2f}")
print(f"Highest-sales category: {top_category} (₹{top_category_sales:,.2f})")
print(f"Highest average-sales customer segment: {top_segment} (₹{segment_avg.iloc[0]:,.2f})")
print(f"Average customer rating: {df['Customer Rating'].mean():.2f}/5")
print(f"Average discount: {df['Discount %'].mean():.2f}%")
print(f"Price vs Sales correlation: {df['Price (INR)'].corr(df['Total Sales (INR)']):.3f}")
print(f"95% CI for mean sales: ₹{ci.loc[ci['Confidence']=='95%', 'Lower'].iloc[0]:,.2f} to ₹{ci.loc[ci['Confidence']=='95%', 'Upper'].iloc[0]:,.2f}")


## 25. Business Recommendations

1. Prioritize inventory and merchandising attention for **Electronics** because it leads total sales.
2. Use **monthly sales forecasts** for inventory and procurement planning, and refresh models as more periods accumulate.
3. Segment promotions by **customer segment** instead of applying one offer universally.
4. Monitor **discount levels jointly with profit**, not sales alone.
5. Improve customer experience and product/service quality to raise the average rating.
6. Monitor category, regional and payment-channel performance through a dashboard.
7. Run randomized **A/B and factorial experiments** before major website or discount changes.
8. Use scenario simulation and optimization for capacity, inventory and campaign-budget decisions.


## 26. Conclusion

This notebook implements the complete Business Analytics workflow described in the supplied Flipkart project report. Transaction data are used for descriptive, statistical, regression, forecasting and business analyses, while simulated/proposed sections are clearly identified where the dataset lacks experimental fields.

The main limitation is that the transaction dataset does not provide website impressions/conversions, marketing spend, or true randomized experimental assignments. These fields would improve future A/B testing and causal analysis.


## 27. Reproducibility Notes

- Random seed: **42**
- Dataset: `flipkart_sales_enriched.csv`
- Historical analysis uses the supplied transaction records.
- A/B testing is simulated because the dataset has no impressions/conversions.
- The factorial section is exploratory when applied to historical observational data.
- The report states that the dataset contains 1,000 records and 17 primary fields; the CSV may contain additional derived fields such as cluster/discount level, which are retained during loading.
